# Photometry Reconciliation

In [1]:
from pathlib import Path
import json, re, pandas as pd
from skyportal_corpus.canonical.document import render_canonical
from skyportal_corpus.extraction_v2.photometry_annotations import merge_photometry_measurements
from skyportal_corpus.extraction_v2.photometry_prose import ProsePhotometryExtractor
from skyportal_corpus.extraction_v2.photometry_rows import PhotometryRowParser
ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
LAG_PATH = ROOT / "notebooks" / "evidence" / "04_photometry_lag.csv"
DETAIL_DIR = ROOT / "data" / "raw" / "skyportal" / "source_detail_20260724"
ARCHIVE_DIR = ROOT / "data" / "raw" / "gcn" / "circulars" / "archive_json" / "20260720_093324" / "extracted" / "archive.json"
lag_evidence = pd.read_csv(LAG_PATH)
print(f"Loaded NB04 evidence rows: {len(lag_evidence):,}")
assert len(lag_evidence) == 7968, f"Expected 7,968 rows, found {len(lag_evidence):,}"
raw_rows = []
for path in sorted(DETAIL_DIR.glob("*/photometry.json")):
    payload = json.loads(path.read_text())["payload"]["data"]
    records = payload.get("photometry", payload.get("data", [])) if isinstance(payload, dict) else payload
    raw_rows.extend({"source_id": path.parent.name, **record} for record in records)
skyportal = pd.DataFrame(raw_rows)
assert len(skyportal) == len(lag_evidence) and skyportal["source_id"].tolist() == lag_evidence["source_id"].tolist()
assert skyportal["filter"].fillna("").tolist() == lag_evidence["band_raw"].fillna("").tolist()
skyportal["circular_id"] = lag_evidence["circular_id"]
skyportal["observation_time_utc"] = pd.Timestamp("1858-11-17", tz="UTC") + pd.to_timedelta(skyportal["mjd"], unit="D")

Loaded NB04 evidence rows: 7,968


In [2]:
SKYPORTAL_BANDS = {"sdssu":"u","sdssg":"g","sdssr":"r","sdssi":"i","sdssz":"z","bessellr":"R","besselli":"I","bessellv":"V","bessellb":"B","2massj":"J","2massh":"H","2massks":"K","uvot::white":"white","uvot::u":"u","uvot::v":"V","uvot::b":"B","uvot::uvw1":"w1","uvot::uvw2":"w2","uvot::uvm2":"m2","ps1::open":"open","gaia::grp":"VT_R","gaia::gbp":"VT_B","gaia::g":"G","gotol":"L","standard::r":"r","standard::i":"i","standard::v":"V","standard::u":"u","ps1::w":"w","ps1::y":"y","ztfr":"r"}
CIRCULAR_BANDS = {"r":"r","r'":"r","SDSS-r":"r","r (AB)":"r","R":"R","Rc":"R","Rc (Vega)":"R","g":"g","g'":"g","i":"i","i'":"i","z":"z","u":"u","U":"u","I":"I","Ic":"I","V":"V","v":"V","B":"B","b":"B","J":"J","H":"H","K":"K","Ks":"K","white":"white","white_FC":"white","white (fc)":"white","u_FC":"u","u (fc)":"u","w1":"w1","uvw1":"w1","w2":"w2","uvw2":"w2","m2":"m2","uvm2":"m2","C":"open","Clear":"open","clear":"open","CR":"open","L":"L","VT_R":"VT_R","VT_B":"VT_B","w":"w"}
def annotation_time(annotation):
    raw = re.sub(r"\s*(?:UTC?|Z)$", "", (annotation.obs_time_raw or "").strip())
    if annotation.obs_time_type == "utc_datetime" and re.search(r"\d{4}[-/]\d{1,2}[-/]\d{1,2}", raw):
        return pd.to_datetime(raw, utc=True, errors="coerce", format="mixed")
    number = re.search(r"\d+(?:\.\d+)?", raw)
    if not number:
        return pd.NaT
    if annotation.obs_time_type == "mjd":
        return pd.Timestamp("1858-11-17", tz="UTC") + pd.to_timedelta(float(number.group()), unit="D")
    if annotation.obs_time_type == "jd":
        return pd.to_datetime(float(number.group()), unit="D", origin="julian", utc=True)
    return pd.NaT
row_parser, prose_parser = PhotometryRowParser(), ProsePhotometryExtractor(); measurement_rows = []
matched = skyportal[skyportal["circular_id"].notna()].copy()
matched["circular_id"] = matched["circular_id"].astype(int)
for circular_id in sorted(matched["circular_id"].unique()):
    record = json.loads((ARCHIVE_DIR / f"{circular_id}.json").read_text())
    created_on = pd.to_datetime(record["createdOn"], unit="ms", utc=True).isoformat()
    document = render_canonical(circular_id, record["subject"], record["body"], record.get("eventId"), created_on, record.get("submitter"))
    annotations = merge_photometry_measurements(row_parser.extract(document), prose_parser.extract(document))
    for ordinal, annotation in enumerate(annotations):
        measurement_rows.append({"circular_id":circular_id,"ordinal":ordinal,"measurement_type":annotation.measurement_type,"mag_circular":float(annotation.magnitude_or_limit),"band_circular":annotation.photometric_band,"instrument_circular":annotation.instrument,"circular_time":annotation.obs_time_raw,"absolute_time":annotation_time(annotation),"span_start":annotation.span_start})
measurements = pd.DataFrame(measurement_rows)

## 1. The Question

When the same photometric measurement appears in a GCN circular and SkyPortal, did its magnitude or band label change during transcription? The state layer must distinguish a repeated observation from a genuinely new fact. We therefore locate correspondences without using magnitude as a matching key, then compare the values.

## 2. One Measurement, Two Records

A single matched pair makes the comparison concrete. The table preserves each source representation rather than normalizing it for display.

In [3]:
matched = matched.reset_index(names="sp_row"); matched["measurement_type"] = matched["mag"].notna().map({True:"detection", False:"upper_limit"}); matched["mag_skyportal"] = matched["mag"].fillna(matched["limiting_mag"]); matched["family"] = matched["filter"].map(SKYPORTAL_BANDS)
measurements = measurements.reset_index(names="ann_row"); measurements["family"] = measurements["band_circular"].map(CIRCULAR_BANDS)
candidates = matched.merge(measurements, on=["circular_id", "measurement_type"], suffixes=("_sp", "_circ"))
candidates["time_delta_s"] = (candidates["absolute_time"] - candidates["observation_time_utc"]).abs().dt.total_seconds()
close = candidates[candidates["time_delta_s"].le(300)].copy(); close["band_penalty"] = close["family_sp"].ne(close["family_circ"]).astype(int)
time_matches, used_annotations = {}, set()
for row in close.sort_values(["time_delta_s", "band_penalty", "sp_row", "ann_row"], kind="mergesort").itertuples():
    if row.sp_row not in time_matches and row.ann_row not in used_annotations:
        time_matches[row.sp_row] = row.ann_row; used_annotations.add(row.ann_row)
remaining_sp = matched[~matched["sp_row"].isin(time_matches)].sort_values(["circular_id", "measurement_type", "family", "observation_time_utc", "sp_row"])
remaining_circular = measurements[~measurements["ann_row"].isin(used_annotations)].sort_values(["circular_id", "measurement_type", "family", "span_start"])
group_keys = ["circular_id", "measurement_type", "family"]
for frame in (remaining_sp, remaining_circular):
    frame["group_n"] = frame.groupby(group_keys, dropna=False)[group_keys[0]].transform("size"); frame["rank"] = frame.groupby(group_keys, dropna=False).cumcount()
ordered = remaining_sp.merge(remaining_circular, on=group_keys + ["group_n", "rank"], suffixes=("_sp", "_circ"))
pair_map = {**time_matches, **dict(zip(ordered["sp_row"], ordered["ann_row"]))}; measurement_by_id, pair_rows = measurements.set_index("ann_row"), []
for row in matched.itertuples():
    ann_row = pair_map.get(row.sp_row); annotation = measurement_by_id.loc[ann_row] if ann_row is not None else None
    pair_rows.append({"source_id":row.source_id,"circular_id":row.circular_id,"observation_time_utc":row.observation_time_utc,"mag_skyportal":row.mag_skyportal,"mag_circular":annotation["mag_circular"] if annotation is not None else pd.NA,"mag_difference":row.mag_skyportal-annotation["mag_circular"] if annotation is not None else pd.NA,"band_skyportal":row.filter,"band_circular":annotation["band_circular"] if annotation is not None else pd.NA,"bands_agree":row.filter==annotation["band_circular"] if annotation is not None else pd.NA,"instrument_skyportal":row.instrument_name,"instrument_circular":annotation["instrument_circular"] if annotation is not None else pd.NA,"match_status":"time_match" if row.sp_row in time_matches else "family_order_match" if ann_row is not None else "not_found","circular_time":annotation["circular_time"] if annotation is not None else pd.NA})
pairs = pd.DataFrame(pair_rows).sort_values(["source_id", "circular_id", "observation_time_utc"], kind="mergesort")
example_pool = pairs[(pairs["match_status"] != "not_found") & pairs["mag_difference"].abs().le(1e-9) & pairs["instrument_circular"].notna()]
if example_pool.empty: example_pool = pairs[pairs["match_status"] != "not_found"]
example = example_pool.sort_values(["source_id", "circular_id", "observation_time_utc"], kind="mergesort").iloc[0]
comparison = pd.DataFrame([["magnitude",example["mag_circular"],example["mag_skyportal"]],["band",example["band_circular"],example["band_skyportal"]],["instrument",example["instrument_circular"],example["instrument_skyportal"]],["observation time",example["circular_time"],example["observation_time_utc"]]], columns=["field", "circular value", "SkyPortal value"])
print(comparison.to_string(index=False))

           field                 circular value                     SkyPortal value
       magnitude                           19.9                                19.9
            band                              U                             uvot::u
      instrument                           UVOT                                 GCN
observation time 65.3 minutes after the trigger 2025-07-28 00:34:55.000416181+00:00


## 3. Do the Magnitudes Agree?

The comparison covers every pair located by time or by an unambiguous equal-size passband sequence. Referenced SkyPortal rows without a defensible circular measurement remain explicitly unmatched.

In [4]:
found = pairs[pairs["match_status"] != "not_found"].copy()
magnitude_pairs = found.dropna(subset=["mag_skyportal", "mag_circular"])
exact = magnitude_pairs["mag_difference"].abs().le(1e-9)
summary = pd.DataFrame([{
    "pairs compared": len(magnitude_pairs),
    "exact magnitude": int(exact.sum()),
    "different magnitude": int((~exact).sum()),
    "not found in circular": int((pairs["match_status"] == "not_found").sum()),
    "median absolute difference": magnitude_pairs["mag_difference"].abs().median(),
}])
print(summary.round(4).to_string(index=False))

 pairs compared  exact magnitude  different magnitude  not found in circular  median absolute difference
            627              487                  140                    331                         0.0


## 4. Do the Band Labels Agree?

The first table shows the most frequent raw label transitions. The second counts literal label equality without applying a normalization map.

In [5]:
band_pairs = found.dropna(subset=["band_circular", "band_skyportal"])
top_bands = (band_pairs.groupby(["band_circular", "band_skyportal"], dropna=False).size().rename("n").reset_index().sort_values(["n", "band_circular", "band_skyportal"], ascending=[False, True, True], kind="mergesort").head(15))
band_agreement = pd.DataFrame([
    {"result": "identical raw labels", "n": int(band_pairs["bands_agree"].eq(True).sum())},
    {"result": "different raw labels", "n": int(band_pairs["bands_agree"].eq(False).sum())},
])
print("Most common circular to SkyPortal band labels")
print(top_bands.to_string(index=False))
print("\nRaw band-label agreement")
print(band_agreement.to_string(index=False))

Most common circular to SkyPortal band labels
band_circular band_skyportal   n
            r          sdssr 102
            i          sdssi  54
            z          sdssz  52
            R       bessellr  43
            g          sdssg  34
            C      ps1::open  27
         VT_R      gaia::grp  27
         VT_B      gaia::gbp  25
           Rc       bessellr  24
            J         2massj  18
           r'          sdssr  16
        white    uvot::white  16
            V       bessellv  14
            v        uvot::v  11
            L          gotol  10

Raw band-label agreement
              result   n
identical raw labels   0
different raw labels 620


## 5. What This Means

Many located pairs preserve the numerical magnitude, while a meaningful subset changes through calibration or transcription. Raw band labels are generally not preserved because circular shorthand becomes a namespaced SkyPortal filter. Merging the two stores by concatenation would overcount observations. A transformed record should therefore retain an `interpreted_from` relation rather than being asserted as a byte-identical `duplicate_of`.

In [6]:
EXPORT_COLUMNS = ["source_id","circular_id","observation_time_utc","mag_skyportal","mag_circular","mag_difference","band_skyportal","band_circular","bands_agree","instrument_skyportal","instrument_circular","match_status"]
EXPORT_PATH = ROOT / "notebooks" / "evidence" / "05_photometry_pairs.csv"
exported_pairs = pairs[EXPORT_COLUMNS].copy()
exported_pairs.to_csv(EXPORT_PATH, index=False)
print(f"Evidence shape: {exported_pairs.shape}")
print(exported_pairs.head(3).to_string(index=False))

Evidence shape: (958, 12)
source_id  circular_id                observation_time_utc  mag_skyportal mag_circular mag_difference band_skyportal band_circular bands_agree instrument_skyportal instrument_circular       match_status
  2025gcz        39889 2025-03-27 21:28:15.000384285+00:00           14.1         14.1           -0.0       bessellr             R       False                  GCN                None family_order_match
  2025gcz        39890 2025-03-27 21:19:07.996800138+00:00           14.2         <NA>           <NA>       bessellv          <NA>        <NA>                  GCN                <NA>          not_found
  2025gcz        39890 2025-03-27 21:19:07.996800138+00:00           15.8         <NA>           <NA>       bessellr          <NA>        <NA>                  GCN                <NA>          not_found


## 6. Why the Magnitudes Differ

Within almost every band pair, the magnitude offset is constant to floating-point precision, as shown by standard deviations that round to zero. The offsets are consistent with tabulated Vega-to-AB conversions, although this notebook does not independently verify them against a published table. SkyPortal therefore appears to normalize each measurement to a canonical photometric system: it renames the band and converts the value rather than copying the published magnitude. The exception is `R -> bessellr`, with standard deviation 0.457 over 29 rows, whereas `Rc -> bessellr` has 0.000 over 15; this remains an observation.

In [7]:
OFFSET_PATH = ROOT / "notebooks" / "evidence" / "05_photometry_pairs.csv"
offset_pairs = pd.read_csv(OFFSET_PATH)
changed_pairs = offset_pairs[
    offset_pairs["mag_difference"].notna()
    & offset_pairs["mag_difference"].abs().gt(1e-9)
    & offset_pairs["mag_difference"].abs().le(100)
]
offset_table = (
    changed_pairs.groupby(["band_circular", "band_skyportal"], dropna=False)["mag_difference"]
    .agg(count="size", median_difference="median", standard_deviation="std").reset_index()
    .sort_values(["count", "band_circular", "band_skyportal"], ascending=[False, True, True], kind="mergesort").head(15)
)
print(offset_table.round(6).to_string(index=False))

band_circular band_skyportal  count  median_difference  standard_deviation
            R       bessellr     29           0.193287            0.456597
           Rc       bessellr     15           0.193287            0.000000
            J         2massj      9           0.899302            0.000000
            V       bessellv      9           0.010087            0.000000
        Clear       bessellr      7           0.193287            0.000000
            B       bessellb      4          -0.102140            0.000000
            H         2massh      4           1.372842            0.000000
    Rc (Vega)       bessellr      4           0.193287            0.000000
        white    uvot::white      4           0.854312            0.000000
           Ic       besselli      3           0.441312            0.000000
            u        uvot::u      3           1.011608            0.000000
            v        uvot::v      3          -0.000286            0.000000
        21.70       besse